# KOBIS 일일 박스오피스 순위 가져오기 (Google Colab)

**목표**: 영화진흥위원회(KOBIS) API로 어제 날짜의 일일 박스오피스 TOP10을 받아와 CSV로 저장합니다.

## 실습 흐름
1. 설정 + API 호출 + 결과 출력 + CSV 저장까지 한 셀에서 진행

> ⚠️ **에러가 나면 셀을 하나씩 다시 실행하지 말고, 메뉴에서 "런타임 → 모두 실행(Run all)"을 눌러 처음부터 다시 실행하세요.**
> (중간 셀만 다시 실행하면 이전 실행 결과가 남아있어 오류 원인 파악이 더 어려워집니다.)

이 실습은 그래프를 그리지 않아서 한글 폰트 설치 셀이 필요 없습니다.


## 셀 1. 설정 + 데이터 수집 + 저장

In [ ]:
import requests
import csv
from datetime import datetime, timedelta

# 1) 여기에 발급받은 키를 넣으세요
KOBIS_KEY = "여기에_발급받은_인증키_입력"

# 2) 조회할 날짜 (직접 지정하려면 "20260814"처럼 문자열로)
TARGET_DATE = (datetime.now() - timedelta(days=1)).strftime("%Y%m%d")

# 3) API 요청 주소 만들기
url = "http://kobis.or.kr/kobisopenapi/webservice/rest/boxoffice/searchDailyBoxOfficeList.json"
params = {"key": KOBIS_KEY, "targetDt": TARGET_DATE}

# 4) 요청 보내고 결과 받기
response = requests.get(url, params=params)
data = response.json()

# 5) 필요한 부분만 꺼내기
box_office_list = data["boxOfficeResult"]["dailyBoxOfficeList"]

print(f"=== {TARGET_DATE} 일일 박스오피스 TOP 10 ===\n")
for movie in box_office_list:
    rank = movie["rank"]
    title = movie["movieNm"]
    audience_today = movie["audiCnt"]
    audience_total = movie["audiAcc"]
    sales_today = movie["salesAmt"]

    print(f"{rank}위 | {title}")
    print(f"   당일 관객수: {int(audience_today):,}명")
    print(f"   누적 관객수: {int(audience_total):,}명")
    print(f"   당일 매출액: {int(sales_today):,}원")
    print("-" * 40)

# 6) CSV로 저장 (코랩 세션의 /content/ 폴더에 저장됩니다)
csv_filename = f"boxoffice_{TARGET_DATE}.csv"
with open(csv_filename, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["순위", "영화제목", "당일관객수", "누적관객수", "당일매출액"])
    for movie in box_office_list:
        writer.writerow([movie["rank"], movie["movieNm"],
                          movie["audiCnt"], movie["audiAcc"], movie["salesAmt"]])

print(f"\n저장 완료: {csv_filename} (왼쪽 폴더 아이콘 -> 파일 목록에서 다운로드 가능)")

# from google.colab import files
# files.download(csv_filename)  # 바로 내 컴퓨터로 받고 싶으면 주석 해제
